In [1]:
import os

print("Files in /kaggle/working:")
print(os.listdir("/kaggle/working"))

Files in /kaggle/working:
[]


In [2]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/saranyanya/civil-unrest/gru_14day_model.keras
/kaggle/input/datasets/saranyanya/civil-unrest/random_forest_temporal_model.joblib
/kaggle/input/datasets/saranyanya/civil-unrest/rf_features.json
/kaggle/input/datasets/saranyanya/civil-unrest/xlm_roberta_event_detector/config.json
/kaggle/input/datasets/saranyanya/civil-unrest/xlm_roberta_event_detector/model.safetensors
/kaggle/input/datasets/saranyanya/civil-unrest/xlm_roberta_event_detector/tokenizer.json
/kaggle/input/datasets/saranyanya/civil-unrest/xlm_roberta_event_detector/tokenizer_config.json


In [3]:
import os
import joblib

BASE = "/kaggle/input/datasets/saranyanya/civil-unrest"

print("XLM-R:", os.path.exists(f"{BASE}/xlm_roberta_event_detector/model.safetensors"))
print("GRU:", os.path.exists(f"{BASE}/gru_14day_model.keras"))
print("RF:", os.path.exists(f"{BASE}/random_forest_temporal_model.joblib"))
print("Features:", os.path.exists(f"{BASE}/rf_features.json"))

XLM-R: True
GRU: True
RF: True
Features: True


In [4]:
import spacy

nlp = spacy.load("en_core_web_sm")

print("spaCy loaded successfully.")

spaCy loaded successfully.


In [6]:
from datasets import load_dataset

# Search/load a Hindi-English event-related dataset candidate
print("Checking available datasets...")

Checking available datasets...


In [7]:
from datasets import load_dataset
import pandas as pd
import os

# Reload the original event dataset
event_dataset = load_dataset("adamnik/event_detection_dataset")

print(event_dataset)
print(event_dataset["train"].column_names)

# Convert to pandas
df = event_dataset["train"].to_pandas()

print("\nOriginal dataset:")
print(df.shape)
print(df["label"].value_counts())

print("\nSample:")
print(df.head())

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

event_detection.csv:   0%|          | 0.00/299k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1743 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 1743
    })
})
['sentence', 'label']

Original dataset:
(1743, 2)
label
1    1019
0     724
Name: count, dtype: int64

Sample:
                                            sentence  label
0  The truth is that you pay for your lifestyle i...      0
1  Israeli soldiers Sunday violently suppressed a...      1
2  A day after police arrested hundreds of protes...      1
3  Ethiopian police said the pistols were discove...      1
4  The Iraqi government forces and Kurdistan's Pe...      1


In [8]:
!pip -q install sentencepiece sacremoses

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

TRANSLATION_MODEL = "Helsinki-NLP/opus-mt-en-hi"

translator_tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL)
translator_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATION_MODEL)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
translator_model = translator_model.to(device)
translator_model.eval()

print("Translation model loaded.")
print("Device:", device)

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  306MB            

... (2 outputs left out)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Translation model loaded.
Device: cuda


In [9]:
from tqdm.auto import tqdm
import pandas as pd
import torch

def translate_batch(texts, batch_size=8):
    translations = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]

        inputs = translator_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)

        with torch.no_grad():
            outputs = translator_model.generate(
                **inputs,
                max_length=256,
                num_beams=4
            )

        decoded = translator_tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )

        translations.extend(decoded)

    return translations


# Translate all 1,743 sentences
df["sentence_hi"] = translate_batch(
    df["sentence"].tolist(),
    batch_size=8
)

print("\nTranslation complete.")
print("Rows:", len(df))

print("\nEnglish:")
print(df.iloc[1]["sentence"])

print("\nHindi:")
print(df.iloc[1]["sentence_hi"])

print("\nDataset:")
print(df.head())

  0%|          | 0/218 [00:00<?, ?it/s]


Translation complete.
Rows: 1743

English:
Israeli soldiers Sunday violently suppressed a peaceful anti-wall protest in the town of Beit Jala near Bethlehem, according to local sources

Hindi:
इस्राएल के सैनिकों ने रविवार को बेतलेहेम के पास बेतबा शहर में शांति के विरोध को ज़ोर से दबा दिया

Dataset:
                                            sentence  label  \
0  The truth is that you pay for your lifestyle i...      0   
1  Israeli soldiers Sunday violently suppressed a...      1   
2  A day after police arrested hundreds of protes...      1   
3  Ethiopian police said the pistols were discove...      1   
4  The Iraqi government forces and Kurdistan's Pe...      1   

                                         sentence_hi  
0  सच तो यह है कि आप अपने जीने के तरीके के लिए घं...  
1  इस्राएल के सैनिकों ने रविवार को बेतलेहेम के पा...  
2  पुलिस के सैकड़ों विरोधियों के एक दिन के बाद......  
3  इथियोपिया की पुलिस ने कहा कि एक सार्वजनिक परिव...  
4  इराकी सरकार सेना और Kughhhrg की सेना एक अस

In [10]:
# Create bilingual examples
english_df = df[["sentence", "label"]].copy()
english_df["language"] = "en"

hindi_df = df[["sentence_hi", "label"]].copy()
hindi_df = hindi_df.rename(columns={"sentence_hi": "sentence"})
hindi_df["language"] = "hi"

# Combine English + Hindi
bilingual_df = pd.concat(
    [english_df, hindi_df],
    ignore_index=True
)

# Remove empty translations
bilingual_df = bilingual_df[
    bilingual_df["sentence"].notna() &
    (bilingual_df["sentence"].str.strip() != "")
].reset_index(drop=True)

print("Bilingual dataset shape:", bilingual_df.shape)

print("\nLanguage distribution:")
print(bilingual_df["language"].value_counts())

print("\nLabel distribution:")
print(bilingual_df["label"].value_counts())

print("\nFirst Hindi examples:")
display(
    bilingual_df[bilingual_df["language"] == "hi"].head(10)
)

Bilingual dataset shape: (3486, 3)

Language distribution:
language
en    1743
hi    1743
Name: count, dtype: int64

Label distribution:
label
1    2038
0    1448
Name: count, dtype: int64

First Hindi examples:


,sentence,label,language
1743,सच तो यह है कि आप अपने जीने के तरीके के लिए घं...,0,hi
1744,इस्राएल के सैनिकों ने रविवार को बेतलेहेम के पा...,1,hi
1745,पुलिस के सैकड़ों विरोधियों के एक दिन के बाद......,1,hi
1746,इथियोपिया की पुलिस ने कहा कि एक सार्वजनिक परिव...,1,hi
1747,इराकी सरकार सेना और Kughhhrg की सेना एक अस्थाय...,1,hi
1748,तुम्हारे लिए वहाँ एक संदेश है अगर तुम देख रहे हो.,0,hi
1749,"रिक शॉ नाम के हाई स्कूल में एक मित्र था, लेकिन...",0,hi
1750,इसके अलावा और जानकारी अल-बुब्बी सेनाओं और सरका...,1,hi
1751,अल-बुब के सदस्यों ने 22 अप्रैल की रात को सरकार...,1,hi
1752,रविवार की शुरूआत में पुलिस ने एक प्रसिद्ध कर्त...,1,hi


In [11]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temporary
train_df, temp_df = train_test_split(
    bilingual_df,
    test_size=0.30,
    stratify=bilingual_df[["label", "language"]],
    random_state=42
)

# Second split: 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[["label", "language"]],
    random_state=42
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain language distribution:")
print(train_df["language"].value_counts())

print("\nValidation language distribution:")
print(val_df["language"].value_counts())

print("\nTest language distribution:")
print(test_df["language"].value_counts())

print("\nTest label distribution:")
print(test_df["label"].value_counts())

Train: (2440, 3)
Validation: (523, 3)
Test: (523, 3)

Train language distribution:
language
en    1220
hi    1220
Name: count, dtype: int64

Validation language distribution:
language
hi    262
en    261
Name: count, dtype: int64

Test language distribution:
language
en    262
hi    261
Name: count, dtype: int64

Test label distribution:
label
1    306
0    217
Name: count, dtype: int64


In [12]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(
    train_df[["sentence", "label"]].reset_index(drop=True)
)

val_dataset = Dataset.from_pandas(
    val_df[["sentence", "label"]].reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df[["sentence", "label"]].reset_index(drop=True)
)

bilingual_dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

print(bilingual_dataset)
print("\nColumns:", bilingual_dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 2440
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 523
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 523
    })
})

Columns: ['sentence', 'label']


In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "xlm-roberta-base"

bilingual_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return bilingual_tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_bilingual_dataset = bilingual_dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_bilingual_dataset)
print("\nTokenized columns:")
print(tokenized_bilingual_dataset["train"].column_names)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/2440 [00:00<?, ? examples/s]

Map:   0%|          | 0/523 [00:00<?, ? examples/s]

Map:   0%|          | 0/523 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 2440
    })
    validation: Dataset({
        features: ['sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 523
    })
    test: Dataset({
        features: ['sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 523
    })
})

Tokenized columns:
['sentence', 'label', 'input_ids', 'attention_mask']


In [14]:
from transformers import AutoModelForSequenceClassification

bilingual_model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=2
)

print("Bilingual XLM-RoBERTa model loaded.")
print("Parameters:",
      sum(p.numel() for p in bilingual_model.parameters()))

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Bilingual XLM-RoBERTa model loaded.
Parameters: 278045186


In [15]:
from datasets import DatasetDict

# Re-tokenize the bilingual datasets
def tokenize_function(examples):
    return bilingual_tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_bilingual_dataset = bilingual_dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_bilingual_dataset)

Map:   0%|          | 0/2440 [00:00<?, ? examples/s]

Map:   0%|          | 0/523 [00:00<?, ? examples/s]

Map:   0%|          | 0/523 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 2440
    })
    validation: Dataset({
        features: ['sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 523
    })
    test: Dataset({
        features: ['sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 523
    })
})


In [17]:
import numpy as np
import torch
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


training_args = TrainingArguments(
    output_dir="/kaggle/working/xlm_roberta_bilingual",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=bilingual_model,
    args=training_args,
    train_dataset=tokenized_bilingual_dataset["train"],
    eval_dataset=tokenized_bilingual_dataset["validation"],
    compute_metrics=compute_metrics
)

print("Starting bilingual XLM-R training...")
trainer.train()

Starting bilingual XLM-R training...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.220167,0.227770,0.982792,1.000000,0.970588,0.985075
2,0.117100,0.177090,0.986616,1.000000,0.977124,0.988430


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=306, training_loss=0.24447005004865524, metrics={'train_runtime': 148.4392, 'train_samples_per_second': 32.875, 'train_steps_per_second': 2.061, 'total_flos': 320995487539200.0, 'train_loss': 0.24447005004865524, 'epoch': 2.0})

In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)
import numpy as np

# Overall test set
test_output = trainer.predict(
    tokenized_bilingual_dataset["test"]
)

test_predictions = np.argmax(
    test_output.predictions,
    axis=-1
)

test_labels = np.array(
    tokenized_bilingual_dataset["test"]["label"]
)

def print_metrics(name, y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )

    accuracy = accuracy_score(y_true, y_pred)

    print(f"\n{name}")
    print("-" * 40)
    print(f"Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision : {precision:.4f} ({precision*100:.2f}%)")
    print(f"Recall    : {recall:.4f} ({recall*100:.2f}%)")
    print(f"F1 Score  : {f1:.4f} ({f1*100:.2f}%)")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# Overall
print_metrics(
    "OVERALL TEST SET",
    test_labels,
    test_predictions
)


# English-only
english_indices = [
    i for i, lang in enumerate(test_df["language"])
    if lang == "en"
]

english_labels = test_labels[english_indices]
english_predictions = test_predictions[english_indices]

print_metrics(
    "ENGLISH TEST SET",
    english_labels,
    english_predictions
)


# Hindi-only
hindi_indices = [
    i for i, lang in enumerate(test_df["language"])
    if lang == "hi"
]

hindi_labels = test_labels[hindi_indices]
hindi_predictions = test_predictions[hindi_indices]

print_metrics(
    "HINDI TEST SET",
    hindi_labels,
    hindi_predictions
)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



OVERALL TEST SET
----------------------------------------
Accuracy  : 0.9828 (98.28%)
Precision : 0.9806 (98.06%)
Recall    : 0.9902 (99.02%)
F1 Score  : 0.9854 (98.54%)
Confusion Matrix:
[[211   6]
 [  3 303]]

ENGLISH TEST SET
----------------------------------------
Accuracy  : 0.9962 (99.62%)
Precision : 0.9935 (99.35%)
Recall    : 1.0000 (100.00%)
F1 Score  : 0.9967 (99.67%)
Confusion Matrix:
[[108   1]
 [  0 153]]

HINDI TEST SET
----------------------------------------
Accuracy  : 0.9693 (96.93%)
Precision : 0.9677 (96.77%)
Recall    : 0.9804 (98.04%)
F1 Score  : 0.9740 (97.40%)
Confusion Matrix:
[[103   5]
 [  3 150]]


In [19]:
import os

BILINGUAL_MODEL_PATH = "/kaggle/working/xlm_roberta_bilingual_event_detector"

trainer.save_model(BILINGUAL_MODEL_PATH)
bilingual_tokenizer.save_pretrained(BILINGUAL_MODEL_PATH)

print("Bilingual XLM-R model saved.")
print("\nFiles:")

for file in os.listdir(BILINGUAL_MODEL_PATH):
    print(file)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Bilingual XLM-R model saved.

Files:
model.safetensors
tokenizer.json
config.json
training_args.bin
tokenizer_config.json


In [20]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

BILINGUAL_MODEL_PATH = "/kaggle/working/xlm_roberta_bilingual_event_detector"

# Load saved model
test_tokenizer = AutoTokenizer.from_pretrained(
    BILINGUAL_MODEL_PATH
)

test_model = AutoModelForSequenceClassification.from_pretrained(
    BILINGUAL_MODEL_PATH
)

test_model.eval()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

test_model = test_model.to(device)

print("Saved bilingual model loaded.")
print("Device:", device)


def predict_event(text):
    inputs = test_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = test_model(**inputs)
        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )[0]

    event_probability = probabilities[1].item()
    prediction = int(event_probability >= 0.5)

    return prediction, event_probability


test_sentences = [
    # Hindi event
    "दिल्ली में हजारों छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया।",

    # Hindi non-event
    "आज दिल्ली में मौसम बहुत अच्छा है और लोग पार्क में घूम रहे हैं।",

    # Hindi event
    "मुंबई में मजदूरों ने वेतन बढ़ाने की मांग को लेकर हड़ताल शुरू कर दी।",

    # English event
    "Thousands of students protested in Delhi against the new education policy.",

    # English non-event
    "The weather is pleasant today and people are walking in the park."
]

for text in test_sentences:
    prediction, probability = predict_event(text)

    print("\n" + "=" * 60)
    print("Text:", text)
    print("Detected Event:", "YES" if prediction == 1 else "NO")
    print(f"Event Probability: {probability * 100:.2f}%")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Saved bilingual model loaded.
Device: cuda

Text: दिल्ली में हजारों छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया।
Detected Event: YES
Event Probability: 99.99%

Text: आज दिल्ली में मौसम बहुत अच्छा है और लोग पार्क में घूम रहे हैं।
Detected Event: NO
Event Probability: 0.05%

Text: मुंबई में मजदूरों ने वेतन बढ़ाने की मांग को लेकर हड़ताल शुरू कर दी।
Detected Event: YES
Event Probability: 99.98%

Text: Thousands of students protested in Delhi against the new education policy.
Detected Event: YES
Event Probability: 99.96%

Text: The weather is pleasant today and people are walking in the park.
Detected Event: NO
Event Probability: 0.04%


In [22]:
import re

EVENT_TRIGGERS = {
    "protest": [
        # English
        "protest", "protested", "protesting", "protester", "protesters",
        "demonstration", "demonstrate", "demonstrated", "demonstrating",
        "march", "marched", "marching",
        "rally", "rallied", "rallying",
        "sit-in", "sit in", "occupation", "occupy", "occupied",

        # Hindi
        "विरोध", "विरोध प्रदर्शन", "विरोध प्रदर्शन किया",
        "प्रदर्शन", "प्रदर्शन किया", "प्रदर्शन कर",
        "धरना", "धरने", "धरना दिया",
        "मार्च", "मार्च किया", "रैली", "रैली किया",
        "आंदोलन", "आंदोलन किया"
    ],

    "strike": [
        # English
        "strike", "struck", "striking",
        "walkout", "walked out",
        "boycott", "boycotted", "boycotting",

        # Hindi
        "हड़ताल", "हड़ताल की", "हड़ताल पर",
        "काम बंद", "कामबंदी",
        "बहिष्कार", "बहिष्कार किया"
    ],

    "clash": [
        # English
        "clash", "clashed", "clashing",
        "fighting", "fight", "fought",
        "confrontation", "confronted",
        "violence", "violent",
        "battle", "battled",

        # Hindi
        "झड़प", "झड़प हुई", "झड़पों",
        "संघर्ष", "संघर्ष हुआ",
        "मुठभेड़", "मुठभेड़ हुई",
        "टकराव", "टकराव हुआ",
        "हिंसा", "हिंसक",
        "लड़ाई", "लड़ाई हुई"
    ],

    "riot": [
        # English
        "riot", "rioted", "rioting", "riots",
        "looting", "looted",

        # Hindi
        "दंगा", "दंगे", "दंगों",
        "दंगा हुआ", "दंगे हुए",
        "लूटपाट", "लूटपाट की"
    ],

    "unrest": [
        # English
        "unrest", "uprising", "uprisings",
        "revolt", "revolted", "revolting",
        "rebellion", "rebellions",
        "rebelled", "rebels",

        # Hindi
        "अशांति", "अशांत",
        "विद्रोह", "विद्रोह हुआ",
        "बगावत", "बगावत हुई",
        "उपद्रव", "उपद्रव हुआ"
    ],

    "attack": [
        # English
        "attack", "attacked", "attacking",
        "assault", "assaulted",
        "bombed", "bombing",
        "explosion", "exploded",

        # Hindi
        "हमला", "हमला किया", "हमले",
        "आक्रमण", "आक्रमण किया",
        "बमबारी", "बमबारी की",
        "विस्फोट", "विस्फोट हुआ"
    ],

    "arrest": [
        # English
        "arrest", "arrested", "arresting",
        "detain", "detained", "detaining",
        "detention",

        # Hindi
        "गिरफ्तार", "गिरफ्तारी",
        "पकड़ा", "पकड़े",
        "हिरासत", "हिरासत में"
    ],

    "abduction": [
        # English
        "abduct", "abducted", "abducting",
        "kidnap", "kidnapped", "kidnapping",

        # Hindi
        "अपहरण", "अपहरण किया",
        "अपहरणकर्ता", "अपहृत"
    ],

    "suppression": [
        # English
        "suppress", "suppressed", "suppressing",
        "crackdown", "cracked down",
        "disperse", "dispersed", "dispersing",

        # Hindi
        "दमन", "दमन किया",
        "कार्रवाई", "कार्रवाई की",
        "दमनकारी",
        "तितर-बितर", "तितर-बितर किया",
        "कड़ी कार्रवाई"
    ]
}


def detect_event_type_trigger(text):
    """
    Detect civil-unrest event type and trigger
    for both Hindi and English.
    """

    text_lower = text.lower()

    # Longer phrases first to avoid partial matches
    candidates = []

    for event_type, triggers in EVENT_TRIGGERS.items():
        for trigger in triggers:
            if trigger.lower() in text_lower:
                candidates.append(
                    (len(trigger), event_type, trigger)
                )

    if not candidates:
        return {
            "event_type": None,
            "trigger": None
        }

    # Prefer the longest matching phrase
    candidates.sort(reverse=True)

    _, event_type, trigger = candidates[0]

    return {
        "event_type": event_type,
        "trigger": trigger
    }

In [23]:
test_texts = [
    "Thousands of students protested in Delhi yesterday.",
    "Workers went on strike in Mumbai demanding higher wages.",
    "Police clashed with protesters in Paris.",
    "दिल्ली में हजारों छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया।",
    "मुंबई में मजदूरों ने वेतन बढ़ाने की मांग को लेकर हड़ताल शुरू कर दी।",
    "पुलिस और प्रदर्शनकारियों के बीच झड़प हुई।",
    "शहर में दंगे भड़क उठे।",
    "सरकार ने प्रदर्शनकारियों पर कड़ी कार्रवाई की।"
]

for text in test_texts:
    result = detect_event_type_trigger(text)

    print("\nText:", text)
    print("Event Type:", result["event_type"])
    print("Trigger:", result["trigger"])


Text: Thousands of students protested in Delhi yesterday.
Event Type: protest
Trigger: protested

Text: Workers went on strike in Mumbai demanding higher wages.
Event Type: strike
Trigger: strike

Text: Police clashed with protesters in Paris.
Event Type: protest
Trigger: protesters

Text: दिल्ली में हजारों छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया।
Event Type: protest
Trigger: विरोध प्रदर्शन किया

Text: मुंबई में मजदूरों ने वेतन बढ़ाने की मांग को लेकर हड़ताल शुरू कर दी।
Event Type: strike
Trigger: हड़ताल

Text: पुलिस और प्रदर्शनकारियों के बीच झड़प हुई।
Event Type: protest
Trigger: प्रदर्शन

Text: शहर में दंगे भड़क उठे।
Event Type: riot
Trigger: दंगे

Text: सरकार ने प्रदर्शनकारियों पर कड़ी कार्रवाई की।
Event Type: suppression
Trigger: कड़ी कार्रवाई


In [24]:
!pip -q install transformers sentencepiece

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    pipeline
)
import torch

NER_MODEL = "Davlan/xlm-roberta-base-ner-hrl"

ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL)
ner_model = AutoModelForTokenClassification.from_pretrained(NER_MODEL)

ner_device = 0 if torch.cuda.is_available() else -1

multilingual_ner = pipeline(
    "ner",
    model=ner_model,
    tokenizer=ner_tokenizer,
    aggregation_strategy="simple",
    device=ner_device
)

print("Multilingual NER loaded.")
print("Device:", "GPU" if ner_device == 0 else "CPU")

config.json:   0%|          | 0.00/980 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Multilingual NER loaded.
Device: GPU


In [25]:
test_texts = [
    "Thousands of students protested in Delhi yesterday.",
    "Police clashed with protesters in Mumbai.",
    "दिल्ली में हजारों छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया।",
    "मुंबई में मजदूरों ने वेतन बढ़ाने की मांग को लेकर हड़ताल शुरू कर दी।",
    "पुलिस और प्रदर्शनकारियों के बीच झड़प पटना में हुई।"
]

for text in test_texts:
    print("\n" + "=" * 60)
    print("TEXT:", text)

    entities = multilingual_ner(text)

    for entity in entities:
        print(
            f"{entity['word']} "
            f"| {entity['entity_group']} "
            f"| {entity['score']:.3f}"
        )


TEXT: Thousands of students protested in Delhi yesterday.
Delhi | LOC | 0.999

TEXT: Police clashed with protesters in Mumbai.
Mumbai | LOC | 1.000

TEXT: दिल्ली में हजारों छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया।
दिल्ली | LOC | 1.000

TEXT: मुंबई में मजदूरों ने वेतन बढ़ाने की मांग को लेकर हड़ताल शुरू कर दी।
मुंबई | LOC | 1.000

TEXT: पुलिस और प्रदर्शनकारियों के बीच झड़प पटना में हुई।
पटना | LOC | 1.000


In [26]:
# Install dateparser if needed
!pip -q install dateparser

import re
import dateparser
from dateparser.search import search_dates


# Hindi + English temporal expressions
HINDI_TIME_PATTERNS = [
    r"आज",
    r"कल",
    r"कल रात",
    r"कल सुबह",
    r"कल शाम",
    r"कल दोपहर",
    r"आज रात",
    r"आज सुबह",
    r"आज शाम",
    r"आज दोपहर",
    r"पिछले दिन",
    r"पिछले सप्ताह",
    r"पिछले हफ्ते",
    r"पिछले महीने",
    r"अगले दिन",
    r"अगले सप्ताह",
    r"अगले हफ्ते",
    r"अगले महीने",
    r"कुछ दिन पहले",
    r"कुछ सप्ताह पहले",
    r"कुछ महीने पहले",
    r"पिछले साल",
    r"अगले साल",
    r"सोमवार",
    r"मंगलवार",
    r"बुधवार",
    r"गुरुवार",
    r"शुक्रवार",
    r"शनिवार",
    r"रविवार",
    r"\d+\s*दिन पहले",
    r"\d+\s*दिन बाद",
    r"\d+\s*सप्ताह पहले",
    r"\d+\s*सप्ताह बाद",
    r"\d+\s*महीने पहले",
    r"\d+\s*महीने बाद"
]

ENGLISH_TIME_PATTERNS = [
    r"\btoday\b",
    r"\byesterday\b",
    r"\btomorrow\b",
    r"\blast night\b",
    r"\bthis morning\b",
    r"\bthis evening\b",
    r"\blast week\b",
    r"\bnext week\b",
    r"\blast month\b",
    r"\bnext month\b",
    r"\blast year\b",
    r"\bnext year\b",
    r"\b\d+\s+days?\s+ago\b",
    r"\bin\s+\d+\s+days?\b",
    r"\b\d+\s+weeks?\s+ago\b",
    r"\bin\s+\d+\s+weeks?\b"
]


def extract_time(text):
    """
    Extract Hindi or English temporal expressions.
    Returns the detected time phrase.
    """

    # First check explicit Hindi/English expressions
    patterns = HINDI_TIME_PATTERNS + ENGLISH_TIME_PATTERNS

    matches = []

    for pattern in patterns:
        found = re.findall(pattern, text, flags=re.IGNORECASE)

        for item in found:
            matches.append(item)

    if matches:
        # Return the longest match
        return max(matches, key=len)

    # Fallback to dateparser for explicit dates
    try:
        parsed = search_dates(
            text,
            languages=["en", "hi"],
            settings={
                "PREFER_DATES_FROM": "past",
                "RETURN_AS_TIMEZONE_AWARE": False
            }
        )

        if parsed:
            return parsed[0][0]

    except Exception:
        pass

    return None

In [27]:
time_tests = [
    "Students protested in Delhi yesterday.",
    "Workers will strike in Mumbai tomorrow.",
    "Police clashed with protesters last night.",
    "दिल्ली में छात्रों ने कल विरोध प्रदर्शन किया।",
    "मुंबई में मजदूरों ने आज हड़ताल शुरू की।",
    "पुलिस ने कल रात प्रदर्शनकारियों को गिरफ्तार किया।",
    "अगले सप्ताह किसान आंदोलन करने की योजना बना रहे हैं।"
]

for text in time_tests:
    print("\nText:", text)
    print("Detected Time:", extract_time(text))


Text: Students protested in Delhi yesterday.
Detected Time: yesterday

Text: Workers will strike in Mumbai tomorrow.
Detected Time: tomorrow

Text: Police clashed with protesters last night.
Detected Time: last night

Text: दिल्ली में छात्रों ने कल विरोध प्रदर्शन किया।
Detected Time: कल

Text: मुंबई में मजदूरों ने आज हड़ताल शुरू की।
Detected Time: आज

Text: पुलिस ने कल रात प्रदर्शनकारियों को गिरफ्तार किया।
Detected Time: कल रात

Text: अगले सप्ताह किसान आंदोलन करने की योजना बना रहे हैं।
Detected Time: अगले सप्ताह


In [28]:
import re

TARGET_PATTERNS = [
    # English
    r"\bagainst\s+(.+?)(?:\.|,|$)",
    r"\bover\s+(.+?)(?:\.|,|$)",
    r"\bdemanding\s+(.+?)(?:\.|,|$)",
    r"\bdemanded\s+(.+?)(?:\.|,|$)",
    r"\bfor\s+(.+?)(?:\.|,|$)",

    # Hindi
    r"के खिलाफ\s+(.+?)(?:।|,|$)",
    r"के विरुद्ध\s+(.+?)(?:।|,|$)",
    r"की मांग को लेकर\s+(.+?)(?:।|,|$)",
    r"की मांग के लिए\s+(.+?)(?:।|,|$)",
    r"के विरोध में\s+(.+?)(?:।|,|$)",
    r"को लेकर\s+(.+?)(?:।|,|$)",
    r"के लिए\s+(.+?)(?:।|,|$)"
]


def clean_target(target):
    """Clean extracted target text."""
    if not target:
        return None

    target = target.strip()

    # Remove common trailing words/punctuation
    target = re.sub(
        r"\s+(कल|आज|कल रात|आज रात|yesterday|today|tomorrow|last night)$",
        "",
        target,
        flags=re.IGNORECASE
    )

    target = target.strip(" ,.!?।")

    if len(target) < 2:
        return None

    return target


def extract_target(text):
    """
    Extract the target/issue of a civil unrest event
    from Hindi or English text.
    """

    candidates = []

    for pattern in TARGET_PATTERNS:
        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        for match in matches:
            target = clean_target(match)

            if target:
                candidates.append(target)

    if not candidates:
        return None

    # Prefer the shortest reasonable candidate
    # to avoid capturing an entire sentence.
    candidates.sort(key=len)

    return candidates[0]

In [29]:
target_tests = [
    "Students protested in Delhi against the new education policy.",
    "Workers went on strike in Mumbai demanding higher wages.",
    "People demonstrated over rising food prices.",
    "Students protested for better education facilities.",
    "दिल्ली में छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया।",
    "मुंबई में मजदूरों ने वेतन बढ़ाने की मांग को लेकर हड़ताल शुरू कर दी।",
    "लोग बढ़ती महंगाई के विरोध में प्रदर्शन कर रहे हैं।",
    "छात्र बेहतर शिक्षा सुविधाओं के लिए आंदोलन कर रहे हैं।"
]

for text in target_tests:
    print("\nText:", text)
    print("Target:", extract_target(text))


Text: Students protested in Delhi against the new education policy.
Target: the new education policy

Text: Workers went on strike in Mumbai demanding higher wages.
Target: higher wages

Text: People demonstrated over rising food prices.
Target: rising food prices

Text: Students protested for better education facilities.
Target: better education facilities

Text: दिल्ली में छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया।
Target: विरोध प्रदर्शन किया

Text: मुंबई में मजदूरों ने वेतन बढ़ाने की मांग को लेकर हड़ताल शुरू कर दी।
Target: हड़ताल शुरू कर दी

Text: लोग बढ़ती महंगाई के विरोध में प्रदर्शन कर रहे हैं।
Target: प्रदर्शन कर रहे हैं

Text: छात्र बेहतर शिक्षा सुविधाओं के लिए आंदोलन कर रहे हैं।
Target: आंदोलन कर रहे हैं


In [32]:
import re

def detect_event_context(text):
    """
    Detect civil-unrest event context in Hindi and English.
    """

    text_lower = text.lower()

    # --------------------------------------------------
    # CANCELLED
    # --------------------------------------------------
    cancelled_patterns = [
        # English
        r"\bcancelled\b",
        r"\bcanceled\b",
        r"\bcalled off\b",
        r"\bpostponed\b",
        r"\bprevented\b",

        # Hindi
        r"रद्द",
        r"रद्द किया",
        r"रद्द कर दिया",
        r"स्थगित",
        r"रोक दिया",
        r"रोक दिया गया",
        r"रोकने की घोषणा"
    ]

    for pattern in cancelled_patterns:
        if re.search(pattern, text_lower):
            return "CANCELLED"

    # --------------------------------------------------
    # PLANNED / FUTURE
    # --------------------------------------------------
    planned_patterns = [
        # English
        r"\bwill protest\b",
        r"\bwill strike\b",
        r"\bwill demonstrate\b",
        r"\bwill march\b",
        r"\bwill rally\b",
        r"\bplans to protest\b",
        r"\bplan to protest\b",
        r"\bplanning to protest\b",
        r"\bplanning a protest\b",
        r"\bexpected to protest\b",
        r"\bexpected to continue\b",
        r"\bwill continue\b",
        r"\bto protest\b",
        r"\bto demonstrate\b",

        # Hindi
        r"विरोध प्रदर्शन करेंगे",
        r"विरोध प्रदर्शन करने वाले हैं",
        r"प्रदर्शन करेंगे",
        r"प्रदर्शन करने की योजना",
        r"प्रदर्शन करने की तैयारी",
        r"धरना देंगे",
        r"हड़ताल करेंगे",
        r"मार्च करेंगे",
        r"रैली करेंगे",
        r"आंदोलन करेंगे",
        r"जारी रखने की योजना",
        r"जारी रहेगा",
        r"जारी रखने वाले"
    ]

    for pattern in planned_patterns:
        if re.search(pattern, text_lower):
            return "PLANNED"

    # --------------------------------------------------
    # ONGOING / PRESENT
    # --------------------------------------------------
    ongoing_patterns = [
        # English
        r"\bis protesting\b",
        r"\bare protesting\b",
        r"\bis striking\b",
        r"\bare striking\b",
        r"\bis demonstrating\b",
        r"\bare demonstrating\b",
        r"\bis marching\b",
        r"\bare marching\b",
        r"\bis rioting\b",
        r"\bare rioting\b",
        r"\bis ongoing\b",
        r"\bare ongoing\b",
        r"\bright now\b",
        r"\bcurrently\b",
        r"\bat present\b",
        r"\bongoing\b",

        # Hindi
        r"प्रदर्शन कर रहे हैं",
        r"प्रदर्शन कर रहे",
        r"विरोध कर रहे हैं",
        r"विरोध कर रहे",
        r"हड़ताल कर रहे हैं",
        r"हड़ताल पर हैं",
        r"मार्च कर रहे हैं",
        r"रैली कर रहे हैं",
        r"दंगा कर रहे हैं",
        r"जारी है",
        r"जारी हैं",
        r"अभी",
        r"फिलहाल",
        r"वर्तमान में"
    ]

    for pattern in ongoing_patterns:
        if re.search(pattern, text_lower):
            return "ONGOING"

    # --------------------------------------------------
    # PAST
    # --------------------------------------------------
    past_patterns = [
        # English
        r"\byesterday\b",
        r"\blast night\b",
        r"\blast week\b",
        r"\blast month\b",
        r"\bpreviously\b",
        r"\bearlier\b",
        r"\bprotested\b",
        r"\bstruck\b",
        r"\bmarched\b",
        r"\brallied\b",
        r"\bclashed\b",
        r"\brioted\b",
        r"\bwas arrested\b",
        r"\bwere arrested\b",

        # Hindi
        r"कल",
        r"कल रात",
        r"पिछले दिन",
        r"पिछले सप्ताह",
        r"पिछले हफ्ते",
        r"पहले",
        r"पहले ही",
        r"प्रदर्शन किया",
        r"विरोध किया",
        r"हड़ताल की",
        r"मार्च किया",
        r"रैली की",
        r"झड़प हुई",
        r"दंगा हुआ",
        r"गिरफ्तार किया",
        r"गिरफ्तार कर लिया"
    ]

    for pattern in past_patterns:
        if re.search(pattern, text_lower):
            return "PAST"

    return "UNKNOWN"

In [33]:
context_tests = [
    "Students protested in Delhi yesterday.",
    "Workers are protesting in Mumbai right now.",
    "Students will protest in Delhi tomorrow.",
    "The government cancelled the planned demonstration.",
    
    "छात्रों ने कल दिल्ली में विरोध प्रदर्शन किया।",
    "मुंबई में मजदूर अभी हड़ताल कर रहे हैं।",
    "छात्र कल दिल्ली में विरोध प्रदर्शन करेंगे।",
    "सरकार ने नियोजित प्रदर्शन को रद्द कर दिया।"
]

for text in context_tests:
    print("\nText:", text)
    print("Context:", detect_event_context(text))


Text: Students protested in Delhi yesterday.
Context: PAST

Text: Workers are protesting in Mumbai right now.
Context: ONGOING

Text: Students will protest in Delhi tomorrow.
Context: PLANNED

Text: The government cancelled the planned demonstration.
Context: CANCELLED

Text: छात्रों ने कल दिल्ली में विरोध प्रदर्शन किया।
Context: PAST

Text: मुंबई में मजदूर अभी हड़ताल कर रहे हैं।
Context: ONGOING

Text: छात्र कल दिल्ली में विरोध प्रदर्शन करेंगे।
Context: PLANNED

Text: सरकार ने नियोजित प्रदर्शन को रद्द कर दिया।
Context: CANCELLED


In [34]:
# ============================================================
# FINAL BILINGUAL EVENT EXTRACTION PIPELINE
# ============================================================

def extract_complete_event_bilingual(text):
    """
    Complete Hindi + English civil-unrest event extraction.

    Components:
    1. Event type
    2. Trigger
    3. Actor
    4. Location
    5. Time
    6. Target
    7. Context
    """

    # --------------------------------------------------------
    # 1. EVENT TYPE + TRIGGER
    # --------------------------------------------------------
    event_info = detect_event_type_trigger(text)

    # --------------------------------------------------------
    # 2. ACTOR + LOCATION
    # --------------------------------------------------------
    actor = None
    location = None

    try:
        ner_results = multilingual_ner(text)

        actor_candidates = []
        location_candidates = []

        for entity in ner_results:
            label = entity.get("entity_group", "").upper()
            word = entity.get("word", "").strip()

            if not word:
                continue

            # Person / organization / group
            if label in ["PER", "PERSON", "ORG"]:
                actor_candidates.append(word)

            # Geographic entities
            elif label in ["LOC", "LOCATION", "GPE", "FAC"]:
                location_candidates.append(word)

        if actor_candidates:
            actor = actor_candidates[0]

        if location_candidates:
            location = location_candidates[0]

    except Exception as e:
        print("NER warning:", e)

    # --------------------------------------------------------
    # 3. TIME
    # --------------------------------------------------------
    time_value = extract_time(text)

    # --------------------------------------------------------
    # 4. TARGET
    # --------------------------------------------------------
    target = extract_target(text)

    # --------------------------------------------------------
    # 5. CONTEXT
    # --------------------------------------------------------
    context = detect_event_context(text)

    # --------------------------------------------------------
    # FINAL RESULT
    # --------------------------------------------------------
    return {
        "event_type": event_info.get("event_type"),
        "trigger": event_info.get("trigger"),
        "actor": actor,
        "location": location,
        "time": time_value,
        "target": target,
        "context": context
    }


# ============================================================
# TEST CASES
# ============================================================

test_cases = [
    # English
    "Thousands of students protested in Delhi yesterday against the new education policy.",
    
    "Workers are protesting in Mumbai right now demanding higher wages.",
    
    "Students will demonstrate in Delhi tomorrow against rising tuition fees.",

    # Hindi
    "दिल्ली में हजारों छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया।",

    "मुंबई में मजदूर अभी वेतन बढ़ाने की मांग को लेकर हड़ताल कर रहे हैं।",

    "छात्र कल दिल्ली में बढ़ती फीस के खिलाफ प्रदर्शन करेंगे।"
]


for i, text in enumerate(test_cases, 1):

    print("\n" + "=" * 70)
    print(f"TEST {i}")
    print("=" * 70)

    print("Text:")
    print(text)

    result = extract_complete_event_bilingual(text)

    print("\nStructured Information:")
    print("Event Type :", result["event_type"])
    print("Trigger    :", result["trigger"])
    print("Actor      :", result["actor"])
    print("Location   :", result["location"])
    print("Time       :", result["time"])
    print("Target     :", result["target"])

    print("\nContext Analysis:")
    print("Context    :", result["context"])


TEST 1
Text:
Thousands of students protested in Delhi yesterday against the new education policy.

Structured Information:
Event Type : protest
Trigger    : protested
Actor      : None
Location   : Delhi
Time       : yesterday
Target     : the new education policy

Context Analysis:
Context    : PAST

TEST 2
Text:
Workers are protesting in Mumbai right now demanding higher wages.


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Structured Information:
Event Type : protest
Trigger    : protesting
Actor      : None
Location   : Mumbai
Time       : now
Target     : higher wages

Context Analysis:
Context    : ONGOING

TEST 3
Text:
Students will demonstrate in Delhi tomorrow against rising tuition fees.

Structured Information:
Event Type : protest
Trigger    : demonstrate
Actor      : None
Location   : Delhi
Time       : tomorrow
Target     : rising tuition fees

Context Analysis:
Context    : PLANNED

TEST 4
Text:
दिल्ली में हजारों छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया।

Structured Information:
Event Type : protest
Trigger    : विरोध प्रदर्शन किया
Actor      : None
Location   : दिल्ली
Time       : None
Target     : विरोध प्रदर्शन किया

Context Analysis:
Context    : PAST

TEST 5
Text:
मुंबई में मजदूर अभी वेतन बढ़ाने की मांग को लेकर हड़ताल कर रहे हैं।

Structured Information:
Event Type : strike
Trigger    : हड़ताल
Actor      : None
Location   : मुंबई
Time       : None
Target     : हड़ताल कर रह

In [35]:
import re

# ============================================================
# IMPROVED TARGET EXTRACTION
# ============================================================

TARGET_PATTERNS = [
    # -------------------------
    # ENGLISH
    # -------------------------
    r"\bagainst\s+(.+?)(?:\.|,|$)",
    r"\bover\s+(.+?)(?:\.|,|$)",
    r"\bdemanding\s+(.+?)(?:\.|,|$)",
    r"\bdemanded\s+(.+?)(?:\.|,|$)",

    # Only use "for" when followed by a meaningful noun phrase
    r"\bfor\s+(?!protest|protesting|strike|striking|demonstration)(.+?)(?:\.|,|$)",

    # -------------------------
    # HINDI
    # -------------------------
    r"के खिलाफ\s+(.+?)(?:।|,|$)",
    r"के विरुद्ध\s+(.+?)(?:।|,|$)",
    r"की मांग को लेकर\s+(.+?)(?:।|,|$)",
    r"की मांग के लिए\s+(.+?)(?:।|,|$)",
    r"के विरोध में\s+(.+?)(?:।|,|$)",

    # Important: capture what comes BEFORE the demand phrase
    r"(.+?)\s+की मांग को लेकर",
]


def clean_target(target):
    if not target:
        return None

    target = target.strip()

    # Remove event-action words accidentally captured
    target = re.sub(
        r"^(विरोध प्रदर्शन|प्रदर्शन|धरना|हड़ताल|आंदोलन|रैली|मार्च)\s*",
        "",
        target,
        flags=re.IGNORECASE
    )

    target = re.sub(
        r"\s+(कर रहे हैं|कर रहे|करेंगे|किया|कर दिया|शुरू कर दी)$",
        "",
        target,
        flags=re.IGNORECASE
    )

    target = target.strip(" ,.!?।")

    if len(target) < 2:
        return None

    return target


def extract_target(text):

    candidates = []

    for pattern in TARGET_PATTERNS:

        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        for match in matches:

            # Some regex groups may return tuples
            if isinstance(match, tuple):
                match = match[0]

            target = clean_target(match)

            if target:
                candidates.append(target)

    if not candidates:
        return None

    # Prefer the most meaningful candidate
    candidates = sorted(
        candidates,
        key=lambda x: len(x),
        reverse=True
    )

    return candidates[0]


# ============================================================
# IMPROVED ACTOR EXTRACTION
# ============================================================

def extract_actor(text):

    # First try multilingual NER
    try:
        ner_results = multilingual_ner(text)

        candidates = []

        for entity in ner_results:

            label = entity.get("entity_group", "").upper()
            word = entity.get("word", "").strip()

            if not word:
                continue

            if label in ["PER", "PERSON", "ORG"]:
                candidates.append(word)

        if candidates:
            return candidates[0]

    except Exception as e:
        print("NER actor warning:", e)

    # --------------------------------------------------------
    # English fallback
    # --------------------------------------------------------

    english_patterns = [
        r"\b(thousands of students)\b",
        r"\b(students)\b",
        r"\b(workers)\b",
        r"\b(protesters)\b",
        r"\b(protesters?)\b",
        r"\b(teachers)\b",
        r"\b(farmers)\b",
        r"\b(residents)\b",
        r"\b(unions?)\b",
        r"\b(activists)\b",
        r"\b(citizens)\b",
        r"\b(staff)\b",
        r"\b(employees)\b"
    ]

    for pattern in english_patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:
            return match.group(1)

    # --------------------------------------------------------
    # Hindi fallback
    # --------------------------------------------------------

    hindi_patterns = [
        r"(हजारों छात्र)",
        r"(हजारों छात्रों)",
        r"(छात्र)",
        r"(छात्रों)",
        r"(मजदूर)",
        r"(मजदूरों)",
        r"(कर्मचारी)",
        r"(कर्मचारियों)",
        r"(किसान)",
        r"(किसानों)",
        r"(प्रदर्शनकारी)",
        r"(कार्यकर्ता)",
        r"(कार्यकर्ताओं)",
        r"(निवासी)",
        r"(निवासियों)"
    ]

    for pattern in hindi_patterns:

        match = re.search(pattern, text)

        if match:
            return match.group(1)

    return None

In [37]:
def extract_complete_event_bilingual(text):

    # Event type + trigger
    event_info = detect_event_type_trigger(text)

    # Actor
    actor = extract_actor(text)

    # Location
    location = None

    try:
        ner_results = multilingual_ner(text)

        location_candidates = []

        for entity in ner_results:

            label = entity.get("entity_group", "").upper()
            word = entity.get("word", "").strip()

            if not word:
                continue

            if label in ["LOC", "LOCATION", "GPE", "FAC"]:
                location_candidates.append(word)

        if location_candidates:
            location = location_candidates[0]

    except Exception:
        pass

    # Time
    time_value = extract_time(text)

    # Target
    target = extract_target(text)

    # Context
    context = detect_event_context(text)

    return {
        "event_type": event_info.get("event_type"),
        "trigger": event_info.get("trigger"),
        "actor": actor,
        "location": location,
        "time": time_value,
        "target": target,
        "context": context
    }

In [38]:
for i, text in enumerate(test_cases, 1):

    print("\n" + "=" * 70)
    print(f"TEST {i}")
    print("=" * 70)

    print("Text:")
    print(text)

    result = extract_complete_event_bilingual(text)

    print("\nStructured Information:")
    print("Event Type :", result["event_type"])
    print("Trigger    :", result["trigger"])
    print("Actor      :", result["actor"])
    print("Location   :", result["location"])
    print("Time       :", result["time"])
    print("Target     :", result["target"])

    print("\nContext Analysis:")
    print("Context    :", result["context"])


TEST 1
Text:
Thousands of students protested in Delhi yesterday against the new education policy.

Structured Information:
Event Type : protest
Trigger    : protested
Actor      : Thousands of students
Location   : Delhi
Time       : yesterday
Target     : the new education policy

Context Analysis:
Context    : PAST

TEST 2
Text:
Workers are protesting in Mumbai right now demanding higher wages.

Structured Information:
Event Type : protest
Trigger    : protesting
Actor      : Workers
Location   : Mumbai
Time       : now
Target     : higher wages

Context Analysis:
Context    : ONGOING

TEST 3
Text:
Students will demonstrate in Delhi tomorrow against rising tuition fees.

Structured Information:
Event Type : protest
Trigger    : demonstrate
Actor      : Students
Location   : Delhi
Time       : tomorrow
Target     : rising tuition fees

Context Analysis:
Context    : PLANNED

TEST 4
Text:
दिल्ली में हजारों छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया।

Structured Information

In [39]:
# ============================================================
# FINAL TARGET EXTRACTION — ENGLISH + HINDI
# ============================================================

def extract_target(text):

    candidates = []

    # ========================================================
    # ENGLISH
    # ========================================================

    english_patterns = [
        r"\bagainst\s+(.+?)(?:\.|,|$)",
        r"\bover\s+(.+?)(?:\.|,|$)",
        r"\bdemanding\s+(.+?)(?:\.|,|$)",
        r"\bdemanded\s+(.+?)(?:\.|,|$)",
        r"\bfor\s+(?!protest|protesting|strike|striking|demonstration)(.+?)(?:\.|,|$)"
    ]

    for pattern in english_patterns:

        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        for match in matches:
            target = match.strip()
            target = target.strip(" ,.!?")

            if len(target) >= 2:
                candidates.append(target)

    # ========================================================
    # HINDI
    # ========================================================

    # Pattern:
    # "नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया"
    # → "नई शिक्षा नीति"
    #
    # Pattern:
    # "वेतन बढ़ाने की मांग को लेकर हड़ताल"
    # → "वेतन बढ़ाने"
    #
    # Pattern:
    # "बढ़ती फीस के खिलाफ प्रदर्शन करेंगे"
    # → "बढ़ती फीस"

    hindi_patterns = [

        # X के खिलाफ ...
        r"([\u0900-\u097F\s]+?)\s+के खिलाफ",

        # X के विरुद्ध ...
        r"([\u0900-\u097F\s]+?)\s+के विरुद्ध",

        # X के विरोध में ...
        r"([\u0900-\u097F\s]+?)\s+के विरोध में",

        # X की मांग को लेकर ...
        r"([\u0900-\u097F\s]+?)\s+की मांग को लेकर",

        # X की मांग के लिए ...
        r"([\u0900-\u097F\s]+?)\s+की मांग के लिए",

        # X को लेकर ...
        r"([\u0900-\u097F\s]+?)\s+को लेकर"
    ]

    for pattern in hindi_patterns:

        matches = re.findall(pattern, text)

        for match in matches:

            target = match.strip()

            # Remove leading sentence/location material
            target = re.sub(
                r"^(दिल्ली में|मुंबई में|दिल्ली के|मुंबई के)\s+",
                "",
                target
            )

            # Remove common actor phrases accidentally included
            target = re.sub(
                r"^(हजारों छात्रों ने|हजारों छात्र|छात्रों ने|छात्र|मजदूरों ने|मजदूर|कर्मचारियों ने|कर्मचारी)\s+",
                "",
                target
            )

            target = target.strip(" ,.!?।")

            if len(target) >= 2:
                candidates.append(target)

    # ========================================================
    # CLEAN RESULTS
    # ========================================================

    cleaned = []

    for target in candidates:

        # Don't return action/event phrases as targets
        if re.fullmatch(
            r"(विरोध प्रदर्शन|प्रदर्शन|हड़ताल|आंदोलन|रैली|मार्च|करेंगे|किया|कर रहे हैं|कर रहे|शुरू कर दी)",
            target,
            flags=re.IGNORECASE
        ):
            continue

        cleaned.append(target)

    if not cleaned:
        return None

    # Prefer the shortest meaningful target.
    # This avoids capturing the entire sentence.
    cleaned.sort(key=len)

    return cleaned[0]

In [41]:
# ============================================================
# TARGET EXTRACTION TEST
# ============================================================

target_tests = [
    "दिल्ली में हजारों छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया.",
    "मुंबई में मजदूर अभी वेतन बढ़ाने की मांग को लेकर हड़ताल कर रहे हैं.",
    "छात्र कल दिल्ली में बढ़ती फीस के खिलाफ प्रदर्शन करेंगे.",
    
    "Thousands of students protested in Delhi yesterday against the new education policy.",
    "Workers are protesting in Mumbai right now demanding higher wages.",
    "Students will demonstrate in Delhi tomorrow against rising tuition fees."
]

for text in target_tests:
    print("\nText:", text)
    print("Target:", extract_target(text))


Text: दिल्ली में हजारों छात्रों ने नई शिक्षा नीति के खिलाफ विरोध प्रदर्शन किया.
Target: नई शिक्षा नीति

Text: मुंबई में मजदूर अभी वेतन बढ़ाने की मांग को लेकर हड़ताल कर रहे हैं.
Target: अभी वेतन बढ़ाने

Text: छात्र कल दिल्ली में बढ़ती फीस के खिलाफ प्रदर्शन करेंगे.
Target: कल दिल्ली में बढ़ती फीस

Text: Thousands of students protested in Delhi yesterday against the new education policy.
Target: the new education policy

Text: Workers are protesting in Mumbai right now demanding higher wages.
Target: higher wages

Text: Students will demonstrate in Delhi tomorrow against rising tuition fees.
Target: rising tuition fees


In [45]:
import os

BILINGUAL_MODEL_PATH = "/kaggle/working/xlm_roberta_bilingual_event_detector"

print("Model exists:", os.path.exists(BILINGUAL_MODEL_PATH))

if os.path.exists(BILINGUAL_MODEL_PATH):
    print("\nFiles:")
    for file in os.listdir(BILINGUAL_MODEL_PATH):
        print(file)

Model exists: True

Files:
model.safetensors
tokenizer.json
config.json
training_args.bin
tokenizer_config.json


In [46]:
import shutil
import os

BILINGUAL_MODEL_PATH = "/kaggle/working/xlm_roberta_bilingual_event_detector"
ZIP_PATH = "/kaggle/working/xlm_roberta_bilingual_event_detector"

zip_file = shutil.make_archive(
    ZIP_PATH,
    "zip",
    BILINGUAL_MODEL_PATH
)

print("Saved successfully:")
print(zip_file)

print(
    "Size:",
    round(os.path.getsize(zip_file) / (1024 * 1024), 2),
    "MB"
)

Saved successfully:
/kaggle/working/xlm_roberta_bilingual_event_detector.zip
Size: 786.82 MB
